# BTC 1m Volume Pattern Checker (dla Google Colab)

**WAŻNE:** Uruchom w Google Colab (działa). Pobiera ostatnie bary 1m BTC z Coinbase API (nie Binance, bo blokuje Colab). Oblicza kolor i sprawdza Twój warunek: zmiana koloru + wyższy volume niż poprzedni bar tego koloru.

In [ ]:
import requests
from datetime import datetime

def get_last_10_1m_bars():
    url = "https://api.exchange.coinbase.com/products/BTC-USD/candles?granularity=60"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        if not isinstance(data, list) or len(data) == 0:
            print("Błąd API lub pusta odpowiedź:", data)
            return []
        data = data[-10:]  # ostatnie 10 barów (Coinbase zwraca od najstarszego)
        bars = []
        for bar in data:
            open_time = datetime.fromtimestamp(bar[0]).strftime('%H:%M')
            o = float(bar[3])  # open
            c = float(bar[4])  # close
            v = float(bar[5])  # volume
            color = "green" if c > o else "red"
            bars.append({"time": open_time, "open": o, "close": c, "volume": v, "color": color})
        return bars
    except Exception as e:
        print("Błąd pobierania:", e)
        return []

def check_volume_pattern(bars):
    if len(bars) < 2:
        return "Za mało danych do analizy."
    for i in range(1, len(bars)):
        curr = bars[i]
        prev = bars[i-1]
        if curr["color"] != prev["color"]:
            last_same_color_vol = 0
            for j in range(i-1, -1, -1):
                if bars[j]["color"] == curr["color"]:
                    last_same_color_vol = bars[j]["volume"]
                    break
            if curr["volume"] > last_same_color_vol:
                return f"WZORZEC WYKRYTY! Bar {curr['time']}: kolor {curr['color']}, volume {curr['volume']:.2f} > poprzedni {last_same_color_vol:.2f}"
    return "Brak wzorca w ostatnich 10 barach 1m."

# === URUCHOMIENIE W COLAB ===
print("=== BTC 1m Volume Pattern Checker (Colab) ===")
bars = get_last_10_1m_bars()
if bars:
    print("Ostatnie 10 barów 1m (najnowszy na końcu):")
    for b in bars:
        print(f"{b['time']} | {b['color']:6} | vol: {b['volume']:.2f}")
    print()
    result = check_volume_pattern(bars)
    print(result)
else:
    print("Nie udało się pobrać danych.")